# MIT 91205 — Day One
## From Raw Data to Insight

This notebook is the Jupyter version of the Day 1 labs (the same style as the lecture: `localhost:8889/notebooks/Day One.ipynb`).

**How to run a cell:** click it, then press `Shift + Enter`.

Run the cells **from top to bottom**. The first code cell loads `df`. Every later cell uses that table.


## Load the sales table

`pandas` is a table library. `read_csv` loads `sample-sales.csv` into a **DataFrame** named `df` (the usual name for a pandas table).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb

df = pd.read_csv("sample-sales.csv")
df.head()


,Order ID,Order Date,Region,Product,Units Sold,Unit Price,Total Revenue
0,1001,2024-01-07,East,Laptop,5,1200,6000
1,1002,2024-01-21,North,Laptop,3,1200,3600
2,1003,2024-01-11,West,Monitor,3,300,900
3,1004,2024-01-22,North,Monitor,2,300,600
4,1005,2024-01-28,North,Laptop,5,1200,6000


## Lab 1 — Meet the data

Look at **shape** (rows, columns), column names, summary stats, and missing values. This is how you check the 5 V's.


In [2]:
print("Shape (rows, columns):", df.shape)
print("Columns:", df.columns.tolist())
print()
print("=== Summary statistics ===")
display(df.describe())
print("=== Missing values per column ===")
df.isnull().sum()


Shape (rows, columns): (1014, 7)
Columns: ['Order ID', 'Order Date', 'Region', 'Product', 'Units Sold', 'Unit Price', 'Total Revenue']

=== Summary statistics ===


,Order ID,Units Sold,Unit Price,Total Revenue
count,1014.000000,1014.000000,1014.000000,1014.000000
mean,1507.500000,3.011834,541.503945,1663.175542
std,292.860888,1.417650,429.616937,1670.773276
min,1001.000000,1.000000,25.000000,25.000000
25%,1254.250000,2.000000,300.000000,300.000000
50%,1507.500000,3.000000,500.000000,1200.000000
75%,1760.750000,4.000000,1200.000000,2400.000000
max,2014.000000,5.000000,1200.000000,6000.000000


=== Missing values per column ===


Order ID         0
Order Date       0
Region           0
Product          0
Units Sold       0
Unit Price       0
Total Revenue    0
dtype: int64

## Lab 2 — Descriptive analytics: what happened?

Descriptive analytics **summarises the past**. It does not explain why.


In [3]:
# 1 Total revenue
total_revenue = df["Total Revenue"].sum()
print(f"Total revenue: ${total_revenue:,.2f}")


Total revenue: $1,686,460.00


In [4]:
# 2 Revenue per region
revenue_by_region = df.groupby("Region")["Total Revenue"].sum()
revenue_by_region.sort_values(ascending=False)


Region
North    530150
South    408080
East     400580
West     347650
Name: Total Revenue, dtype: int64

### 3. Top 5 products by revenue

This is the cell from the lecture slide. Read it as a **pipeline** (left to right, then down the dots):

1. `df.groupby("Product")` — split all 1014 orders into buckets, one bucket per product name.
2. `["Total Revenue"]` — from each bucket, keep only the money column.
3. `.sum()` — add the money inside each bucket. Result: one number per product.
4. `.sort_values(ascending=False)` — put the largest revenue first.
5. `.head(5)` — keep only the first five rows (the winners).

The parentheses around the whole expression let Python continue the line. In Jupyter, writing `top_products` on its own **displays** the table (you do not have to `print` it).


In [5]:
# 3 Top 5 products by revenues
top_products = (df.groupby("Product")["Total Revenue"]
                  .sum()
                  .sort_values(ascending=False)
                  .head(5))
print("\nTop 5 products by revenue:")
top_products



Top 5 products by revenue:


Product
Laptop      1020000
Tablet       442000
Monitor      200400
Keyboard      17460
Mouse          6600
Name: Total Revenue, dtype: int64

**What this means in business language:** Laptop made the most money, then Tablet, then Monitor. Keyboard and Mouse are small. That is *what* happened. Lab 3 asks *why* regions differ.


In [6]:
# 4 Monthly revenue
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Month"] = df["Order Date"].dt.to_period("M")
monthly = df.groupby("Month")["Total Revenue"].sum()
monthly


Month
2024-01    123770
2024-02    122145
2024-03    118285
2024-04    107350
2024-05    147030
2024-06    135360
2024-07    156440
2024-08    142890
2024-09    150370
2024-10    159440
2024-11    161685
2024-12    161695
Freq: M, Name: Total Revenue, dtype: int64

In [7]:
# 5 Bar chart
revenue_by_region.plot(kind="bar", title="Revenue by Region")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.savefig("revenue_by_region.png")
plt.show()


C:\Users\BIZI\AppData\Local\Temp\ipykernel_11932\3527870596.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Lab 3 — Diagnostic analytics: why did it happen?

A **pivot table** is a cross-tab: Region down the side, Product across the top, revenue in the cells. That shows *which product* drives each region.


In [8]:
pivot = df.pivot_table(
    values="Total Revenue",
    index="Region",
    columns="Product",
    aggfunc="sum",
    fill_value=0
)

print("Revenue by region and product:")
display(pivot.round(2))

print("Top product per region:")
for region in pivot.index:
    top = pivot.loc[region].idxmax()
    value = pivot.loc[region].max()
    print(f"  {region}: {top} (${value:,.2f})")

print("\nLaptop as % of each region's revenue:")
(pivot["Laptop"] / pivot.sum(axis=1) * 100).round(1)


Revenue by region and product:


Product,Keyboard,Laptop,Monitor,Mouse,Tablet
Region,,,,,
East,3105,284400,44700,1875,66500
North,6525,355200,47100,1325,120000
South,4455,159600,55800,1725,186500
West,3375,220800,52800,1675,69000


Top product per region:
  East: Laptop ($284,400.00)
  North: Laptop ($355,200.00)
  South: Tablet ($186,500.00)
  West: Laptop ($220,800.00)

Laptop as % of each region's revenue:


Region
East     71.0
North    67.0
South    39.1
West     63.5
dtype: float64

**Insight:** North / East / West are Laptop-heavy. **South is the outlier — Tablets dominate there.** Diagnostic analytics found the pattern. It still cannot say *why* South prefers Tablets (that would need surveys or extra columns).


## Lab 4 — Predictive analytics: a straight-line forecast

We fit `revenue = slope * month + intercept` and extend it 3 months. This is the *idea* of prediction, not a production ML model.


In [9]:
monthly_df = (df.groupby(df["Order Date"].dt.to_period("M"))["Total Revenue"]
                .sum()
                .reset_index())
monthly_df.columns = ["Month", "Revenue"]
monthly_df["MonthIndex"] = range(len(monthly_df))

slope, intercept = np.polyfit(monthly_df["MonthIndex"], monthly_df["Revenue"], 1)
print(f"Trend line: Revenue = {slope:,.2f} * month + {intercept:,.2f}")

next_months = [len(monthly_df), len(monthly_df) + 1, len(monthly_df) + 2]
for m in next_months:
    pred = slope * m + intercept
    print(f"  Predicted revenue for month {m}: ${pred:,.2f}")

plt.figure(figsize=(8, 4))
plt.plot(monthly_df["MonthIndex"], monthly_df["Revenue"], "o-", label="Actual")
plt.plot(monthly_df["MonthIndex"], slope * monthly_df["MonthIndex"] + intercept,
         "r--", label="Trend")
plt.plot(next_months, [slope * m + intercept for m in next_months],
         "g^--", label="Forecast")
plt.xlabel("Month index")
plt.ylabel("Revenue ($)")
plt.legend()
plt.tight_layout()
plt.savefig("revenue_forecast.png")
plt.show()


Trend line: Revenue = 4,492.59 * month + 115,829.10
  Predicted revenue for month 12: $169,740.15
  Predicted revenue for month 13: $174,232.74
  Predicted revenue for month 14: $178,725.33


C:\Users\BIZI\AppData\Local\Temp\ipykernel_11932\4067366459.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Lab 5 — Partitioning and replication

We cannot run Hadoop on a laptop, so we **simulate** the ideas:

- **Partition** = split the table by Region (like splitting a file into chunks).
- **Replicate** = keep 3 copies of each chunk (HDFS default).
- **Fail a node** = delete one copy; recover from a surviving copy.


In [10]:
partitions = {}
for region, chunk in df.groupby("Region"):
    partitions[region] = chunk.reset_index(drop=True)
    print(f"Partition '{region}': {len(chunk)} rows")

REPLICATION_FACTOR = 3
replicas = {region: [partitions[region].copy() for _ in range(REPLICATION_FACTOR)]
            for region in partitions}

print(f"\nEach partition has {REPLICATION_FACTOR} replicas:")
for region in replicas:
    print(f"  '{region}': {len(replicas[region])} copies")

print("\n>>> Simulating: primary node of 'North' fails <<<")
replicas["North"].pop(0)
print(f"'North' now has {len(replicas['North'])} healthy replicas")

recovered = replicas["North"][0]
print(f"Recovered {len(recovered)} rows for North from surviving replica.")
recovered.head(1)


Partition 'East': 224 rows
Partition 'North': 297 rows
Partition 'South': 278 rows
Partition 'West': 215 rows

Each partition has 3 replicas:
  'East': 3 copies
  'North': 3 copies
  'South': 3 copies
  'West': 3 copies

>>> Simulating: primary node of 'North' fails <<<
'North' now has 2 healthy replicas
Recovered 297 rows for North from surviving replica.


,Order ID,Order Date,Region,Product,Units Sold,Unit Price,Total Revenue,Month
0,1002,2024-01-21,North,Laptop,3,1200,3600,2024-01


## Lab 6 — CAP theorem in practice

During a **network partition** you cannot have both:

- **C**onsistency (every node shows the same value)
- **A**vailability (every request gets an answer)

**Scenario B = CP** (banks / ATMs): refuse the stale node.  
**Scenario C = AP** (social likes): serve stale data so the app stays up.


In [11]:
node_a = {"balance": 100}
node_b = {"balance": 100}

def write_balance(node, new_balance):
    node["balance"] = new_balance

def read_balance(node):
    return node["balance"]

print("=== Scenario A: Network healthy ===")
write_balance(node_a, 200)
write_balance(node_b, 200)
print(f"node_a: {read_balance(node_a)}")
print(f"node_b: {read_balance(node_b)}")
print("Both consistent.")

print("\n=== Scenario B: Partition, choose CP (Consistency) ===")
node_a["balance"] = 300
print("node_a:", node_a["balance"])
print("Client reads node_b: ERROR (node_b is unreachable)")

print("\n=== Scenario C: Partition, choose AP (Availability) ===")
print("Client reads node_b:", node_b["balance"], "(stale but available)")
print("node_a has 300; node_b still says 200.")
print("Availability chosen over consistency.")


=== Scenario A: Network healthy ===
node_a: 200
node_b: 200
Both consistent.

=== Scenario B: Partition, choose CP (Consistency) ===
node_a: 300
Client reads node_b: ERROR (node_b is unreachable)

=== Scenario C: Partition, choose AP (Availability) ===
Client reads node_b: 200 (stale but available)
node_a has 300; node_b still says 200.
Availability chosen over consistency.


## Lab 7 — Full analytics lifecycle

Question → load → clean → SQL → chart → recommendation.


In [12]:
print("=" * 50)
print("BUSINESS QUESTION:")
print("  Which region should we expand, and with which product?")
print("=" * 50)

clean = df.drop(columns=["Month"], errors="ignore").dropna()
print(f"After cleaning: {len(clean)} complete rows")

con = duckdb.connect()
con.register("sales", clean)

result = con.execute('''
    SELECT
        Region,
        Product,
        SUM("Total Revenue") AS total_revenue,
        SUM("Units Sold")    AS units,
        SUM("Total Revenue") / SUM("Units Sold") AS revenue_per_unit
    FROM sales
    GROUP BY Region, Product
    ORDER BY total_revenue DESC
''').df()

print("\nTop revenue combinations (region + product):")
display(result.head(10))

top5 = result.head(5)
labels = top5["Region"] + " - " + top5["Product"]
plt.figure(figsize=(8, 4))
plt.barh(labels, top5["total_revenue"])
plt.xlabel("Revenue ($)")
plt.title("Top 5 Region-Product Combinations")
plt.tight_layout()
plt.savefig("top_combinations.png")
plt.show()

top = result.iloc[0]
print("=" * 50)
print("RECOMMENDATION (validate with business):")
print(f"  Focus on {top['Region']} with {top['Product']}.")
print("=" * 50)


BUSINESS QUESTION:
  Which region should we expand, and with which product?
After cleaning: 1014 complete rows



Top revenue combinations (region + product):


,Region,Product,total_revenue,units,revenue_per_unit
0,North,Laptop,355200.0,296.0,1200.0
1,East,Laptop,284400.0,237.0,1200.0
2,West,Laptop,220800.0,184.0,1200.0
3,South,Tablet,186500.0,373.0,500.0
4,South,Laptop,159600.0,133.0,1200.0
5,North,Tablet,120000.0,240.0,500.0
6,West,Tablet,69000.0,138.0,500.0
7,East,Tablet,66500.0,133.0,500.0
8,South,Monitor,55800.0,186.0,300.0
9,West,Monitor,52800.0,176.0,300.0


RECOMMENDATION (validate with business):
  Focus on North with Laptop.


C:\Users\BIZI\AppData\Local\Temp\ipykernel_11932\1901607951.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
